In [58]:
import pandas as pd
import os
import pickle
from pydantic import BaseModel, Field
from typing import List, Dict,Tuple,Set

# SETS

In [59]:
# Define the directory containing the CSV files
CLEWsy_outputs_dir:str= 'datapackage/SETS/CLEWsy_outputs'

In [60]:
# Get a list of all CSV files in the directory
SETS_csv_files:list[str] = [f for f in os.listdir(CLEWsy_outputs_dir) if f.endswith('.csv')]

# Create a dataframe for each CSV file with the corresponding name
SETS:list = {}

for csv_file in SETS_csv_files:
    # Strip the .csv extension to get the dataframe name
    df_name = os.path.splitext(csv_file)[0]
    
    # Read the CSV file into a dataframe
    df_path = os.path.join(CLEWsy_outputs_dir, csv_file)
    SETS[df_name] = pd.read_csv(df_path)
    
    # Optionally, you can print the dataframe name to verify
    print(f"SETS Dataframe created: {df_name}")
print(f"\nTo access the SETS dataframe, type >>> SETS['set_name_str']")

SETS Dataframe created: COMMODITY
SETS Dataframe created: STORAGE
SETS Dataframe created: MODE_OF_OPERATION
SETS Dataframe created: TIMESLICE
SETS Dataframe created: OutputActivityRatio
SETS Dataframe created: SEASON
SETS Dataframe created: DAILYTIMEBRACKET
SETS Dataframe created: InputActivityRatio
SETS Dataframe created: FUEL
SETS Dataframe created: DAYTYPE
SETS Dataframe created: EMISSION
SETS Dataframe created: YEAR
SETS Dataframe created: REGION
SETS Dataframe created: TECHNOLOGY

To access the SETS dataframe, type >>> SETS['set_name_str']


# PARAMETERS

In [61]:
# Define the directory containing the CSV files
REF_parameter_files_dir:str= 'datapackage/Params'

In [62]:
# Get a list of all CSV files in the directory
Params_csv_files:List[str] = [f for f in os.listdir(REF_parameter_files_dir) if f.endswith('.csv')]

# Create a dataframe for each CSV file with the corresponding name
Parameters = {}

for csv_file in Params_csv_files:
    # Strip the .csv extension to get the dataframe name
    df_name = os.path.splitext(csv_file)[0]
    
    # Read the CSV file into a dataframe
    df_path = os.path.join(REF_parameter_files_dir, csv_file)
    Parameters[df_name] = pd.read_csv(df_path)
    
    # Optionally, you can print the dataframe name to verify
    print(f"Parameters Dataframe created: {df_name}")
    
print(f"\nTo access the Parameters dataframe, type >>> Parameters['set_name_str']")

Parameters Dataframe created: ReserveMarginTagFuel
Parameters Dataframe created: DaySplit
Parameters Dataframe created: REMinProductionTarget
Parameters Dataframe created: RETagFuel
Parameters Dataframe created: ModelPeriodEmissionLimit
Parameters Dataframe created: MinStorageCharge
Parameters Dataframe created: OperationalLifeStorage
Parameters Dataframe created: TradeRoute
Parameters Dataframe created: TotalAnnualMaxCapacityInvestment
Parameters Dataframe created: DiscountRateStorage
Parameters Dataframe created: TechnologyToStorage
Parameters Dataframe created: DaysInDayType
Parameters Dataframe created: CapitalCostStorage
Parameters Dataframe created: OperationalLife
Parameters Dataframe created: TotalTechnologyAnnualActivityLowerLimit
Parameters Dataframe created: Conversionls
Parameters Dataframe created: ReserveMargin
Parameters Dataframe created: StorageMaxDischargeRate
Parameters Dataframe created: CapitalCost
Parameters Dataframe created: TechnologyActivityByModeLowerLimit
Pa

## Capacity Factor

In [63]:
Parameters['CapacityFactor']

,REGION,TECHNOLOGY,TIMESLICE,YEAR,VALUE
0,REGION1,DEMAGRDSL,SEA1D,2020,1.0
1,REGION1,DEMAGRDSL,SEA1N,2020,1.0
2,REGION1,DEMAGRDSL,SEA2D,2020,1.0
3,REGION1,DEMAGRDSL,SEA2N,2020,1.0
4,REGION1,DEMAGRDSL,SEA3D,2020,1.0
...,...,...,...,...,...
39179,REGION1,RNWWND,SEA2N,2050,1.0
39180,REGION1,RNWWND,SEA3D,2050,1.0
39181,REGION1,RNWWND,SEA3N,2050,1.0
39182,REGION1,RNWWND,SEA4D,2050,1.0


### 1. Define Pydantic Validation Model

In [64]:
class CapacityFactor(BaseModel):
    REGION: str
    TECHNOLOGY: str
    TIMESLICE: str
    YEAR: int
    VALUE: float = Field(..., alias='VALUE', ge=0, le=1)
    # Discuss with team mates and Taco for additional validation conditions

### 2. Process Nexus Compatible timeslice dictioaries (str, dataframe) and format + validate with pydantic model

In [65]:
def process_timeslices_forNexus(
        timeslice_file_path: str,
        CapacityFactor: BaseModel,
        Capacity_factor_df: pd.DataFrame
    ) -> pd.DataFrame:
    """
    Processes timeslice data from a pickle file, validates it using a Pydantic model, 
    and appends the validated rows to an existing DataFrame.

    Parameters:
    timeslice_file_path (str): Path to the pickle file containing timeslice data.
    CapacityFactor (BaseModel): Pydantic model for validating the data.
    Capacity_factor_df (pd.DataFrame): DataFrame to append the validated data.

    Returns:
    pd.DataFrame: DataFrame with the validated data appended.
    """

    # Load the dictionary from the pickle file
    with open(timeslice_file_path, 'rb') as file:
        _ts_dict_: Dict[str, pd.DataFrame] = pickle.load(file)

    all_rows = []
    for site in _ts_dict_.keys():
        # Convert DataFrame to list of dictionaries
        _data_dict_ = _ts_dict_[site].to_dict(orient='records')

        # Validate the row using the Pydantic model
        rows = [CapacityFactor(**item) for item in _data_dict_]

        # Collect all validated rows
        all_rows.extend(rows)

    # Dump the validated rows to a list of dictionaries
    _rows_dumped_ = [row.model_dump() for row in all_rows]

    # Append the validated rows to the existing DataFrame
    _data_ = pd.concat([Capacity_factor_df, pd.DataFrame(_rows_dumped_)], ignore_index=True)
    Capacity_factor_df = _data_.loc[:, ['REGION', 'TECHNOLOGY', 'YEAR', 'TIMESLICE', 'VALUE']]

    return Capacity_factor_df

### 2.2 Append the newly formatted data to the existing parameter

In [66]:
# Path to the pickle file
new_ts_file_path:str = 'from_linking_tool/new_sites_timeslice'

# Get a list of all CSV files in the directory
ts_files:List[str] = [f for f in os.listdir(new_ts_file_path) if f.endswith('.pkl')]

# Example DataFrame to append to
Capacity_factor_df:pd.DataFrame = Parameters['CapacityFactor']

# Initialize an empty list to store processed DataFrames
dfs:Dict = []

# Process each timeslice file and store the result
for ts in ts_files:
    df = process_timeslices_forNexus(os.path.join(new_ts_file_path, ts), CapacityFactor, Capacity_factor_df)
    dfs.append(df)

# Concatenate all processed DataFrames
Capacity_factor_df_updated:pd.DataFrame = pd.concat(dfs, ignore_index=True)

### 3. Save the updated datafile

In [67]:
# Define the directory containing the CSV files
REF_parameter_updated_files_dir:str= 'datapackage/updated_data_test'
Capacity_factor_df_updated.to_csv(os.path.join(REF_parameter_updated_files_dir,f"CapacityFactor.csv"))

In [68]:
Capacity_factor_df_updated

,REGION,TECHNOLOGY,YEAR,TIMESLICE,VALUE
0,REGION1,DEMAGRDSL,2020,SEA1D,1.000000
1,REGION1,DEMAGRDSL,2020,SEA1N,1.000000
2,REGION1,DEMAGRDSL,2020,SEA2D,1.000000
3,REGION1,DEMAGRDSL,2020,SEA2N,1.000000
4,REGION1,DEMAGRDSL,2020,SEA3D,1.000000
...,...,...,...,...,...
78779,REGION1,PWRSOLB03,2050,SEA2N,0.188932
78780,REGION1,PWRSOLB03,2050,SEA3D,0.305800
78781,REGION1,PWRSOLB03,2050,SEA3N,0.347606
78782,REGION1,PWRSOLB03,2050,SEA4D,0.318369


## Availability Factor

In [69]:
Parameters['AvailabilityFactor']

,REGION,TECHNOLOGY,YEAR,VALUE
0,REGION1,DEMAGRDSL,2020,1.0
1,REGION1,DEMAGRDSL,2021,1.0
2,REGION1,DEMAGRDSL,2022,1.0
3,REGION1,DEMAGRDSL,2023,1.0
4,REGION1,DEMAGRDSL,2024,1.0
...,...,...,...,...
4893,REGION1,RNWWND,2046,1.0
4894,REGION1,RNWWND,2047,1.0
4895,REGION1,RNWWND,2048,1.0
4896,REGION1,RNWWND,2049,1.0


### 1. Define Pydantic Validation Model

In [70]:
class AvailabilityFactor(BaseModel):
    REGION: str
    TECHNOLOGY: str
    YEAR: int
    VALUE: float = Field(..., alias='VALUE', ge=0, le=1)